**Данный ноутбук предназначен для первичного анализа модулей и данных (EDA) библиотеки nflreadpy. Целью является отбор ключевых признаков и датасетов для построения модели предсказания исходов матчей NFL**

In [2]:
import nflreadpy as nfl
import pandas as pd 
import numpy as np 
import warnings
pd.set_option('display.max_columns', None)
warnings.filterwarnings('ignore')

In [3]:
print(dir(nfl))

['__all__', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', '__version__', 'cache', 'clear_cache', 'config', 'downloader', 'get_current_season', 'get_current_week', 'load_combine', 'load_contracts', 'load_depth_charts', 'load_draft_picks', 'load_ff_opportunity', 'load_ff_playerids', 'load_ff_rankings', 'load_ffverse', 'load_ftn_charting', 'load_injuries', 'load_nextgen_stats', 'load_officials', 'load_participation', 'load_pbp', 'load_pfr_advstats', 'load_player_stats', 'load_players', 'load_rosters', 'load_rosters_weekly', 'load_schedules', 'load_snap_counts', 'load_stats', 'load_team_stats', 'load_teams', 'load_trades', 'utils_date', 'version']


In [4]:
schedules = nfl.load_schedules([2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025])

In [5]:
type(schedules)

polars.dataframe.frame.DataFrame

In [6]:
schedules = schedules.to_pandas()

In [7]:
schedules.head()

,game_id,season,game_type,week,gameday,weekday,gametime,away_team,away_score,home_team,home_score,location,result,total,overtime,old_game_id,gsis,nfl_detail_id,pfr,pff,espn,ftn,away_rest,home_rest,away_moneyline,home_moneyline,spread_line,away_spread_odds,home_spread_odds,total_line,under_odds,over_odds,div_game,roof,surface,temp,wind,away_qb_id,home_qb_id,away_qb_name,home_qb_name,away_coach,home_coach,referee,stadium_id,stadium
0,2016_01_CAR_DEN,2016,REG,1,2016-09-08,Thursday,20:30,CAR,20,DEN,21,Home,1,41,0,2016090800,56901,NaN,201609080den,4094.0,400874484,NaN,7,7,-150.0,136.0,-3.0,101.0,-111.0,40.5,-110.0,-100.0,0,outdoors,grass,85.0,10.0,00-0027939,00-0032156,Cam Newton,Trevor Siemian,Ron Rivera,Gary Kubiak,Gene Steratore,DEN00,Sports Authority Field at Mile High
1,2016_01_TB_ATL,2016,REG,1,2016-09-11,Sunday,13:00,TB,31,ATL,24,Home,-7,55,0,2016091100,56902,NaN,201609110atl,4095.0,400874522,NaN,7,7,120.0,-133.0,2.5,-102.0,-108.0,46.5,-105.0,-105.0,1,dome,fieldturf,NaN,NaN,00-0031503,00-0026143,Jameis Winston,Matt Ryan,Dirk Koetter,Dan Quinn,Jeff Triplette,ATL00,Georgia Dome
2,2016_01_BUF_BAL,2016,REG,1,2016-09-11,Sunday,13:00,BUF,7,BAL,13,Home,6,20,0,2016091101,56903,NaN,201609110rav,4096.0,400874510,NaN,7,7,151.0,-167.0,3.0,105.0,-116.0,44.5,-108.0,-102.0,0,outdoors,grass,84.0,8.0,00-0028118,00-0026158,Tyrod Taylor,Joe Flacco,Rex Ryan,John Harbaugh,Brad Allen,BAL00,M&T Bank Stadium
3,2016_01_CHI_HOU,2016,REG,1,2016-09-11,Sunday,13:00,CHI,14,HOU,23,Home,9,37,0,2016091102,56904,NaN,201609110htx,4097.0,400874511,NaN,7,7,210.0,-235.0,5.5,-107.0,-103.0,43.0,100.0,-110.0,0,closed,fieldturf,NaN,NaN,00-0024226,00-0029682,Jay Cutler,Brock Osweiler,John Fox,Bill O'Brien,Peter Morelli,HOU00,NRG Stadium
4,2016_01_GB_JAX,2016,REG,1,2016-09-11,Sunday,13:00,GB,27,JAX,23,Home,-4,50,0,2016091103,56905,NaN,201609110jax,4098.0,400874486,NaN,7,7,-179.0,161.0,-3.5,-100.0,-110.0,47.0,-110.0,-100.0,0,outdoors,grass,90.0,3.0,00-0023459,00-0031407,Aaron Rodgers,Blake Bortles,Mike McCarthy,Gus Bradley,Carl Cheffers,JAX00,EverBank Field


In [8]:
schedules.columns

Index(['game_id', 'season', 'game_type', 'week', 'gameday', 'weekday',
       'gametime', 'away_team', 'away_score', 'home_team', 'home_score',
       'location', 'result', 'total', 'overtime', 'old_game_id', 'gsis',
       'nfl_detail_id', 'pfr', 'pff', 'espn', 'ftn', 'away_rest', 'home_rest',
       'away_moneyline', 'home_moneyline', 'spread_line', 'away_spread_odds',
       'home_spread_odds', 'total_line', 'under_odds', 'over_odds', 'div_game',
       'roof', 'surface', 'temp', 'wind', 'away_qb_id', 'home_qb_id',
       'away_qb_name', 'home_qb_name', 'away_coach', 'home_coach', 'referee',
       'stadium_id', 'stadium'],
      dtype='str')

**Cодержит календарь всех матчей на основании которых строится как baseline, так и главная модель. В данном датафрейме создается колонка 'winner (home)', которая является целевой переменной (см. baseline_model.ipynb)**

In [9]:
play_by_play = nfl.load_pbp(2025).to_pandas()
play_by_play.head(2)

,play_id,game_id,old_game_id,home_team,away_team,season_type,week,posteam,posteam_type,defteam,side_of_field,yardline_100,game_date,quarter_seconds_remaining,half_seconds_remaining,game_seconds_remaining,game_half,quarter_end,drive,sp,qtr,down,goal_to_go,time,yrdln,ydstogo,ydsnet,desc,play_type,yards_gained,shotgun,no_huddle,qb_dropback,qb_kneel,qb_spike,qb_scramble,pass_length,pass_location,air_yards,yards_after_catch,run_location,run_gap,field_goal_result,kick_distance,extra_point_result,two_point_conv_result,home_timeouts_remaining,away_timeouts_remaining,timeout,timeout_team,td_team,td_player_name,td_player_id,posteam_timeouts_remaining,defteam_timeouts_remaining,total_home_score,total_away_score,posteam_score,defteam_score,score_differential,posteam_score_post,defteam_score_post,score_differential_post,no_score_prob,opp_fg_prob,opp_safety_prob,opp_td_prob,fg_prob,safety_prob,td_prob,extra_point_prob,two_point_conversion_prob,ep,epa,total_home_epa,total_away_epa,total_home_rush_epa,total_away_rush_epa,total_home_pass_epa,total_away_pass_epa,air_epa,yac_epa,comp_air_epa,comp_yac_epa,total_home_comp_air_epa,total_away_comp_air_epa,total_home_comp_yac_epa,total_away_comp_yac_epa,total_home_raw_air_epa,total_away_raw_air_epa,total_home_raw_yac_epa,total_away_raw_yac_epa,wp,def_wp,home_wp,away_wp,wpa,vegas_wpa,vegas_home_wpa,home_wp_post,away_wp_post,vegas_wp,vegas_home_wp,total_home_rush_wpa,total_away_rush_wpa,total_home_pass_wpa,total_away_pass_wpa,air_wpa,yac_wpa,comp_air_wpa,comp_yac_wpa,total_home_comp_air_wpa,total_away_comp_air_wpa,total_home_comp_yac_wpa,total_away_comp_yac_wpa,total_home_raw_air_wpa,total_away_raw_air_wpa,total_home_raw_yac_wpa,total_away_raw_yac_wpa,punt_blocked,first_down_rush,first_down_pass,first_down_penalty,third_down_converted,third_down_failed,fourth_down_converted,fourth_down_failed,incomplete_pass,touchback,interception,punt_inside_twenty,punt_in_endzone,punt_out_of_bounds,punt_downed,punt_fair_catch,kickoff_inside_twenty,kickoff_in_endzone,kickoff_out_of_bounds,kickoff_downed,kickoff_fair_catch,fumble_forced,fumble_not_forced,fumble_out_of_bounds,solo_tackle,safety,penalty,tackled_for_loss,fumble_lost,own_kickoff_recovery,own_kickoff_recovery_td,qb_hit,rush_attempt,pass_attempt,sack,touchdown,pass_touchdown,rush_touchdown,return_touchdown,extra_point_attempt,two_point_attempt,field_goal_attempt,kickoff_attempt,punt_attempt,fumble,complete_pass,assist_tackle,lateral_reception,lateral_rush,lateral_return,lateral_recovery,passer_player_id,passer_player_name,passing_yards,receiver_player_id,receiver_player_name,receiving_yards,rusher_player_id,rusher_player_name,rushing_yards,lateral_receiver_player_id,lateral_receiver_player_name,lateral_receiving_yards,lateral_rusher_player_id,lateral_rusher_player_name,lateral_rushing_yards,lateral_sack_player_id,lateral_sack_player_name,interception_player_id,interception_player_name,lateral_interception_player_id,lateral_interception_player_name,punt_returner_player_id,punt_returner_player_name,lateral_punt_returner_player_id,lateral_punt_returner_player_name,kickoff_returner_player_name,kickoff_returner_player_id,lateral_kickoff_returner_player_id,lateral_kickoff_returner_player_name,punter_player_id,punter_player_name,kicker_player_name,kicker_player_id,own_kickoff_recovery_player_id,own_kickoff_recovery_player_name,blocked_player_id,blocked_player_name,tackle_for_loss_1_player_id,tackle_for_loss_1_player_name,tackle_for_loss_2_player_id,tackle_for_loss_2_player_name,qb_hit_1_player_id,qb_hit_1_player_name,qb_hit_2_player_id,qb_hit_2_player_name,forced_fumble_player_1_team,forced_fumble_player_1_player_id,forced_fumble_player_1_player_name,forced_fumble_player_2_team,forced_fumble_player_2_player_id,forced_fumble_player_2_player_name,solo_tackle_1_team,solo_tackle_2_team,solo_tackle_1_player_id,solo_tackle_2_player_id,solo_tackle_1_player_name,solo_tackle_2_player_name,assist_tackle_1_player_id,assist_tackle_1_player_name,assist_tackle_1_team,assist_tack

In [10]:
play_by_play.columns

Index(['play_id', 'game_id', 'old_game_id', 'home_team', 'away_team',
       'season_type', 'week', 'posteam', 'posteam_type', 'defteam',
       ...
       'out_of_bounds', 'home_opening_kickoff', 'qb_epa', 'xyac_epa',
       'xyac_mean_yardage', 'xyac_median_yardage', 'xyac_success', 'xyac_fd',
       'xpass', 'pass_oe'],
      dtype='str', length=372)

**Объемный, но важный датафрейм - по нему можно будет сделать несколько важных признаков (feature engineering)**

In [11]:
participation = nfl.load_participation(2025).to_pandas()
participation.head(2)

,nflverse_game_id,old_game_id,play_id,possession_team,offense_formation,offense_personnel,defenders_in_box,defense_personnel,number_of_pass_rushers,players_on_play,offense_players,defense_players,n_offense,n_defense,ngs_air_yards,time_to_throw,was_pressure,route,defense_man_zone_type,defense_coverage_type,offense_names,defense_names,offense_positions,defense_positions,offense_numbers,defense_numbers
0,2025_01_DAL_PHI,2025090400,40.0,DAL,NaN,"2 CB, 1 FB, 1 FS, 1 ILB, 2 OLB, 1 SS, 1 TE, 2 WR",0.0,"2 CB, 1 FB, 2 ILB, 1 K, 2 OLB, 1 RB, 1 SS, 1 TE",0.0,00-0039140;00-0039841;00-0031357;00-0040251;00...,00-0031357;00-0040251;00-0038738;00-0037578;00...,00-0039140;00-0039841;00-0038489;00-0039826;00...,11,11,NaN,NaN,False,,,NaN,C.J. Goodwin;Trikweze Bridges;Hunter Luepke;Ju...,Kelee Ringo;Cooper DeJean;Ben VanSumeren;Jerem...,CB;CB;FB;FS;ILB;OLB;OLB;SS;TE;WR;WR,CB;CB;FB;ILB;ILB;K;OLB;OLB;RB;SS;TE,29;25;40;2;32;18;35;14;86;9;19,7;33;43;54;42;4;0;48;28;21;83
1,2025_01_DAL_PHI,2025090400,71.0,DAL,UNDER CENTER,"1 C, 2 G, 1 QB, 1 RB, 2 T, 1 TE, 3 WR",6.0,"3 CB, 2 DT, 1 FS, 2 ILB, 2 OLB, 1 SS",0.0,00-0039348;00-0033878;00-0039888;00-0039841;00...,00-0039348;00-0037243;00-0040007;00-0033077;00...,00-0033878;00-0039888;00-0039841;00-0037073;00...,11,11,NaN,NaN,False,,,NaN,Cooper Beebe;Tyler Smith;Tyler Booker;Dak Pres...,Adoree' Jackson;Quinyon Mitchell;Cooper DeJean...,C;G;G;QB;RB;T;T;TE;WR;WR;WR,CB;CB;CB;DT;DT;FS;ILB;ILB;OLB;OLB;SS,56;73;52;4;33;78;60;87;88;3;1,8;27;33;90;97;32;53;30;3;58;24


In [12]:
snaps = nfl.load_snap_counts(2025).to_pandas()
snaps.head(2)

,game_id,pfr_game_id,season,game_type,week,player,pfr_player_id,position,team,opponent,offense_snaps,offense_pct,defense_snaps,defense_pct,st_snaps,st_pct
0,2025_01_ARI_NO,202509070nor,2025,REG,1,Kelvin Banks,BankKe01,T,NO,ARI,75.0,1.0,0.0,0.0,5.0,0.19
1,2025_01_ARI_NO,202509070nor,2025,REG,1,Cesar Ruiz,RuizCe00,C,NO,ARI,75.0,1.0,0.0,0.0,5.0,0.19


In [13]:
ftn = nfl.load_ftn_charting(2025).to_pandas()
ftn.head(2)

,ftn_game_id,nflverse_game_id,season,week,ftn_play_id,nflverse_play_id,starting_hash,qb_location,n_offense_backfield,n_defense_box,is_no_huddle,is_motion,is_play_action,is_screen_pass,is_rpo,is_trick_play,is_qb_out_of_pocket,is_interception_worthy,is_throw_away,read_thrown,is_catchable_ball,is_contested_ball,is_created_reception,is_drop,is_qb_sneak,n_blitzers,n_pass_rushers,is_qb_fault_sack,date_pulled
0,6734,2025_01_DAL_PHI,2025,1,1106026,40,0,0,0.0,0,False,False,False,False,False,False,False,False,False,0,False,False,False,False,False,0,0,False,2025-12-11 12:41:10.599765+00:00
1,6734,2025_01_DAL_PHI,2025,1,1106027,71,R,U,2.0,6,False,True,False,False,False,False,False,False,False,0,False,False,False,False,False,0,0,False,2025-12-11 12:41:10.599765+00:00


**Три датасета выше не несут в себе особой пользу для модели , так как расстановка защиты для каждого розыгрыша, а также данные был ли play action в качестве розыгрыша слишком тяжелые для обработки и могу  запутать модель**

In [14]:
players = nfl.load_players().to_pandas()
players.head(2)

,gsis_id,display_name,common_first_name,first_name,last_name,short_name,football_name,suffix,esb_id,nfl_id,pfr_id,pff_id,otc_id,espn_id,smart_id,birth_date,position_group,position,ngs_position_group,ngs_position,height,weight,headshot,college_name,college_conference,jersey_number,rookie_season,last_season,latest_team,status,ngs_status,ngs_status_short_description,years_of_experience,pff_position,pff_status,draft_year,draft_round,draft_pick,draft_team
0,00-0028830,Isaako Aaitui,Isaako,Isaako,Aaitui,NaN,NaN,NaN,AAI622937,NaN,AaitIs00,6998,2535,14856,32004141-4962-2937-61ff-017b1804dec6,1987-01-25,DL,NT,NaN,NaN,76.0,307.0,https://static.www.nfl.com/image/private/f_aut...,UNLV,NaN,0,2011,2014,WAS,DEV,NaN,NaN,2,DI,NaN,NaN,NaN,NaN,NaN
1,00-0038389,Israel Abanikanda,Israel,Israel,Abanikanda,I.Abanikanda,Israel,NaN,ABA159567,56008,AbanIs00,122999,10967,4429202,32004142-4115-9567-2e24-0eab29f6a4b9,2002-10-05,RB,RB,RB,RB,70.0,225.0,https://static.www.nfl.com/image/upload/f_auto...,Pittsburgh,Atlantic Coast Conference,30,2023,2026,DAL,ACT,ACT,Active,3,HB,A,2023.0,5.0,143.0,NYJ


**Данные игроков такие как рост, вес, дата рождения не влияют на итог матча**

In [15]:
rosters = nfl.load_rosters(2025).to_pandas()
rosters.head(2)

,season,team,position,depth_chart_position,jersey_number,status,full_name,first_name,last_name,birth_date,height,weight,college,gsis_id,espn_id,sportradar_id,yahoo_id,rotowire_id,pff_id,pfr_id,fantasy_data_id,sleeper_id,years_exp,headshot_url,ngs_position,week,game_type,status_description_abbr,football_name,esb_id,gsis_it_id,smart_id,entry_year,rookie_year,draft_club,draft_number
0,2025,GB,DL,DT,69.0,DEV,Dante Barnett,Dante,Barnett,NaT,NaN,275,NaN,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,https://static.www.nfl.com/image/upload/f_auto...,NaN,19,WC,P03,Dante,BAR591037,58805,32004a55-4435-9919-355b-3e0aef2de56d,2025,2025,NaN,NaN
1,2025,IND,QB,QB,17.0,INA,Philip Rivers,Philip,Rivers,1981-12-08,77.0,228,N.C. State,00-0022942,5529,e47706c7-e14d-41fb-b13b-83a835a1f3bc,6763,3766,1725,RivePh00,8244,331,21,https://static.www.nfl.com/image/private/f_aut...,NaN,18,REG,A01,Philip,RIV651634,28956,32005249-5665-1634-a192-77c9c8e6262e,2004,2004,NYG,4.0


In [16]:
rosters_weekly = nfl.load_rosters_weekly(2025).to_pandas()
rosters_weekly.head(2)

,season,team,position,depth_chart_position,jersey_number,status,full_name,first_name,last_name,birth_date,height,weight,college,gsis_id,espn_id,sportradar_id,yahoo_id,rotowire_id,pff_id,pfr_id,fantasy_data_id,sleeper_id,years_exp,headshot_url,ngs_position,week,game_type,status_description_abbr,football_name,esb_id,gsis_it_id,smart_id,entry_year,rookie_year,draft_club,draft_number
0,2025,GB,DL,DT,69.0,DEV,Dante Barnett,Dante,Barnett,NaT,NaN,275.0,NaN,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,https://static.www.nfl.com/image/upload/f_auto...,NaN,16,REG,P03,Dante,BAR591037,58805,32004a55-4435-9919-355b-3e0aef2de56d,2025,2025,NaN,NaN
1,2025,GB,DL,DT,69.0,DEV,Dante Barnett,Dante,Barnett,NaT,NaN,275.0,NaN,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,https://static.www.nfl.com/image/upload/f_auto...,NaN,17,REG,P03,Dante,BAR591037,58805,32004a55-4435-9919-355b-3e0aef2de56d,2025,2025,NaN,NaN


In [17]:
rosters.columns.equals(rosters_weekly.columns)

True

**Из двух датафреймов будет использован только rosters_weekly, т.к он предоставляет ростер на текущую игровую неделю, на предстоящий матч, а не на момент конца сезона**

In [18]:
depth = nfl.load_depth_charts(2025).to_pandas()
depth.head(2)

,dt,team,player_name,espn_id,gsis_id,pos_grp_id,pos_grp,pos_id,pos_name,pos_abb,pos_slot,pos_rank
0,2026-03-14T07:32:09Z,ARI,Josh Sweat,3693166,00-0034381,16,Base 4-3 D,11,Left Defensive End,LDE,1,1
1,2026-03-14T07:32:09Z,ARI,Roy Lopez,4040805,00-0036568,16,Base 4-3 D,24,Left Defensive Tackle,LDT,2,1


In [19]:
depth['pos_abb'].unique()

<ArrowStringArray>
[ 'LDE',  'LDT',  'RDT',  'RDE',  'WLB',  'MLB',  'SLB',  'LCB',   'SS',
   'FS',  'RCB',   'NB',   'PK',    'P',    'H',   'PR',   'KR',   'LS',
   'WR',   'LT',   'LG',    'C',   'RG',   'RT',   'QB',   'TE',   'RB',
   'NT', 'LILB', 'RILB',   'FB']
Length: 31, dtype: str

**Вероятно использован не будет, так как не дает четкое разделение на QB1, QB2 что сильно помогло бы модели в предсказании, однако возможно данный датафрейм будет доработан**

In [20]:
injuries = nfl.load_injuries(2025).to_pandas()
injuries.head(2)

,season,season_type,game_type,team,week,gsis_id,position,full_name,first_name,last_name,report_primary_injury,report_secondary_injury,report_status,practice_primary_injury,practice_secondary_injury,practice_status
0,2025,REG,REG,ARI,1,00-0026190,DE,Calais Campbell,Calais,Campbell,NaN,NaN,NaN,Not injury related - resting player,NaN,Did Not Participate In Practice
1,2025,REG,REG,ARI,1,00-0034346,G,Will Hernandez,William,Hernandez,Knee,NaN,Out,Knee,NaN,Limited Participation in Practice


**В коопе с rosters_weekly, помогает определить какие игроки примут участие в матче**

In [21]:
referee = nfl.load_officials(2025).to_pandas()
referee.head(2)

,game_id,game_key,official_name,position,jersey_number,official_id,season,season_type,week
0,2025090400,59843,Dyrol Prioleau,Field Judge,109,472,2025,REG,1
1,2025090400,59843,Boris Cheek,Side Judge,41,211,2025,REG,1


**Данные о судьях излишни, а имя главного судьи и так содержится в schedules**

In [22]:
player_stats = nfl.load_stats.load_player_stats(2025).to_pandas()
player_stats.head(2)

,player_id,player_name,player_display_name,position,position_group,headshot_url,season,week,season_type,game_id,team,opponent_team,completions,attempts,passing_yards,passing_tds,passing_interceptions,sacks_suffered,sack_yards_lost,sack_fumbles,sack_fumbles_lost,passing_air_yards,passing_yards_after_catch,passing_first_downs,passing_epa,passing_cpoe,passing_2pt_conversions,pacr,passing_10,passing_16,passing_20,passing_40,carries,rushing_yards,rushing_tds,rushing_fumbles,rushing_fumbles_lost,rushing_first_downs,rushing_epa,rushing_2pt_conversions,rushing_10,rushing_12,rushing_20,rushing_40,receptions,targets,receiving_yards,receiving_tds,receiving_fumbles,receiving_fumbles_lost,receiving_air_yards,receiving_yards_after_catch,receiving_first_downs,receiving_epa,receiving_2pt_conversions,receiving_10,receiving_16,receiving_20,receiving_40,racr,target_share,air_yards_share,wopr,special_teams_tds,def_tackles_solo,def_tackles_with_assist,def_tackle_assists,def_tackles_for_loss,def_tackles_for_loss_yards,def_fumbles_forced,def_sacks,def_sack_yards,def_qb_hits,def_interceptions,def_interception_yards,def_pass_defended,def_tds,def_fumbles,def_safeties,def_punt_blocks,def_pat_blocks,def_fg_blocks,def_2pt_atts,def_2pt_made,misc_yards,fumble_recovery_own,fumble_recovery_yards_own,fumble_recovery_opp,fumble_recovery_yards_opp,fumble_recovery_tds,penalties,penalty_yards,fumbles_forced_by_opp,fumbles_not_forced,fumbles_out_of_bounds,fumbles_total,fumbles_lost_total,punt_returns,punt_return_yards,kickoff_returns,kickoff_return_yards,fg_made,fg_att,fg_missed,fg_blocked,fg_long,fg_pct,fg_made_0_19,fg_made_20_29,fg_made_30_39,fg_made_40_49,fg_made_50_59,fg_made_60_,fg_missed_0_19,fg_missed_20_29,fg_missed_30_39,fg_missed_40_49,fg_missed_50_59,fg_missed_60_,fg_made_list,fg_missed_list,fg_blocked_list,fg_made_distance,fg_missed_distance,fg_blocked_distance,pat_made,pat_att,pat_missed,pat_blocked,pat_pct,gwfg_made,gwfg_att,gwfg_missed,gwfg_blocked,gwfg_distance,pt_att,pt_blocked,pt_long,pt_yards,pt_inside_20,pt_out_of_bounds,pt_downed,pt_touchback,pt_fair_caught,pt_returned,pt_return_yards,pt_return_tds,pt_net_yards,fantasy_points,fantasy_points_ppr
0,00-0023459,A.Rodgers,Aaron Rodgers,QB,QB,https://static.www.nfl.com/image/upload/f_auto...,2025,1,REG,2025_01_PIT_NYJ,PIT,NYJ,22,30,244,4,0,4,-26,0,0,139,173,14,10.204755,6.350381,0,1.755396,11,8,5,0,1,-1,0,0,0,0,-1.873902,0,0,0,0,0,0,0,0,0,0,0,0,0,0,NaN,0,0,0,0,0,NaN,0.0,0.0,0.0,0,0,0,0,0,0,0,0.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,NaN,NaN,0,0,0,0,0,0,0,0,0,0,0,0,NaN,NaN,NaN,0,0,0,0,0,0,0,NaN,0,0,0,0,0,0,0,NaN,0,0,0,0,0,0,0,0,0,0,25.66,25.66
1,00-0023853,M.Prater,Matt Prater,K,SPEC,https://static.www.nfl.com/image/upload/f_auto...,2025,1,REG,2025_01_BAL_BUF,BUF,BAL,0,0,0,0,0,0,0,0,0,0,0,0,NaN,NaN,0,NaN,0,0,0,0,0,0,0,0,0,0,NaN,0,0,0,0,0,0,0,0,0,0,0,0,0,0,NaN,0,0,0,0,0,NaN,0.0,0.0,0.0,0,0,0,0,0,0,0,0.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,3,3,0,0,43.0,1.0,0,1,1,1,0,0,0,0,0,0,0,0,25;43;32,NaN,NaN,100,0,0,2,2,0,0,1.0,1,1,0,0,32,0,0,NaN,0,0,0,0,0,0,0,0,0,0,0.00,0.00


**Полноценный подсчет суммарной статистики за сезон или за какой-то период времени для каждого игрока слишком раздует модель. Поэтому статистика, скорее всего, будет подсчитана для квотербеков, так как его статистика хорошо отражает нападение команды**

In [23]:
tm_stats = nfl.load_stats.load_team_stats(2025).to_pandas()
tm_stats.head(2)

,season,week,team,season_type,game_id,opponent_team,completions,attempts,passing_yards,passing_tds,passing_interceptions,sacks_suffered,sack_yards_lost,sack_fumbles,sack_fumbles_lost,passing_air_yards,passing_yards_after_catch,passing_first_downs,passing_epa,passing_cpoe,passing_2pt_conversions,passing_10,passing_16,passing_20,passing_40,carries,rushing_yards,rushing_tds,rushing_fumbles,rushing_fumbles_lost,rushing_first_downs,rushing_epa,rushing_2pt_conversions,rushing_10,rushing_12,rushing_20,rushing_40,receptions,targets,receiving_yards,receiving_tds,receiving_fumbles,receiving_fumbles_lost,receiving_air_yards,receiving_yards_after_catch,receiving_first_downs,receiving_epa,receiving_2pt_conversions,receiving_10,receiving_16,receiving_20,receiving_40,special_teams_tds,def_tackles_solo,def_tackles_with_assist,def_tackle_assists,def_tackles_for_loss,def_tackles_for_loss_yards,def_fumbles_forced,def_sacks,def_sack_yards,def_qb_hits,def_interceptions,def_interception_yards,def_pass_defended,def_tds,def_fumbles,def_safeties,def_punt_blocks,def_pat_blocks,def_fg_blocks,def_2pt_atts,def_2pt_made,misc_yards,fumble_recovery_own,fumble_recovery_yards_own,fumble_recovery_opp,fumble_recovery_yards_opp,fumble_recovery_tds,penalties,penalty_yards,timeouts,fumbles_forced_by_opp,fumbles_not_forced,fumbles_out_of_bounds,fumbles_total,fumbles_lost_total,punt_returns,punt_return_yards,kickoff_returns,kickoff_return_yards,fg_made,fg_att,fg_missed,fg_blocked,fg_long,fg_pct,fg_made_0_19,fg_made_20_29,fg_made_30_39,fg_made_40_49,fg_made_50_59,fg_made_60_,fg_missed_0_19,fg_missed_20_29,fg_missed_30_39,fg_missed_40_49,fg_missed_50_59,fg_missed_60_,fg_made_list,fg_missed_list,fg_blocked_list,fg_made_distance,fg_missed_distance,fg_blocked_distance,pat_made,pat_att,pat_missed,pat_blocked,pat_pct,gwfg_made,gwfg_att,gwfg_missed,gwfg_blocked,gwfg_distance,pt_att,pt_blocked,pt_long,pt_yards,pt_inside_20,pt_out_of_bounds,pt_downed,pt_touchback,pt_fair_caught,pt_returned,pt_return_yards,pt_return_tds,pt_net_yards
0,2025,1,ARI,REG,2025_01_ARI_NO,NO,21,29,163,2,0,5,-33,0,0,166,100,11,1.524799,-1.733811,0,7,2,2,1,27,146,0,0,0,5,1.158540,0,4,3,1,1,21,29,163,2,0,0,166,100,11,8.777653,0,7,2,2,1,0,36,2,36,3,10,0,1.0,6.0,2,0,0,10,0,0,0,0,0,0,0,0,0,0,0,0,0,0,9,54,0,0,0,0,0,0,3,30,3,73,2,3,0,1,50.0,0.666667,0,0,0,1,1,0,0,0,0,0,0,0,42;50,NaN,46,92,0,46,2,2,0,0,1.0,0,0,0,0,0,4,0,60.0,188,1,1,0,0,1,2,18,0,170
1,2025,1,ATL,REG,2025_01_TB_ATL,TB,27,42,298,1,0,1,-9,0,0,288,189,16,9.624095,-5.938343,0,14,4,3,1,28,69,1,1,0,4,-5.766494,0,1,1,0,0,27,42,298,1,0,0,288,189,16,11.561699,0,14,4,3,1,0,27,1,25,2,3,0,1.0,8.0,3,0,0,7,0,0,0,0,0,0,0,0,0,1,-4,0,0,0,8,55,6,0,1,0,1,0,2,29,5,125,2,3,1,0,41.0,0.666667,0,0,1,1,0,0,0,0,0,1,0,0,41;36,44,NaN,77,44,0,2,2,0,0,1.0,0,0,0,0,0,3,0,59.0,140,2,1,0,0,1,1,54,0,86


In [24]:
tm_stats.columns

Index(['season', 'week', 'team', 'season_type', 'game_id', 'opponent_team',
       'completions', 'attempts', 'passing_yards', 'passing_tds',
       ...
       'pt_yards', 'pt_inside_20', 'pt_out_of_bounds', 'pt_downed',
       'pt_touchback', 'pt_fair_caught', 'pt_returned', 'pt_return_yards',
       'pt_return_tds', 'pt_net_yards'],
      dtype='str', length=138)

**Очень полезный датафрейм, потенциальный заменитель play_by_play (см. выше) так как уже несет в себе статистические метрики для каждой команды**

In [25]:
pfr = nfl.load_pfr_advstats(2025).to_pandas()
pfr.head(2)

,game_id,pfr_game_id,season,week,game_type,team,opponent,pfr_player_name,pfr_player_id,passing_drops,passing_drop_pct,receiving_drop,receiving_drop_pct,passing_bad_throws,passing_bad_throw_pct,times_sacked,times_blitzed,times_hurried,times_hit,times_pressured,times_pressured_pct,def_times_blitzed,def_times_hurried,def_times_hitqb
0,2025_01_DAL_PHI,202509040phi,2025,1,REG,PHI,DAL,Jalen Hurts,HurtJa00,1.0,0.050,NaN,NaN,0.0,0.000,1.0,5.0,1.0,4.0,6.0,0.182,NaN,NaN,NaN
1,2025_01_DAL_PHI,202509040phi,2025,1,REG,DAL,PHI,Dak Prescott,PresDa01,3.0,0.091,NaN,NaN,3.0,0.091,0.0,5.0,2.0,1.0,3.0,0.086,NaN,NaN,NaN


In [26]:
pfr.isna().sum()

game_id                    0
pfr_game_id                0
season                     0
week                       0
game_type                  0
team                       0
opponent                   0
pfr_player_name            0
pfr_player_id              0
passing_drops              0
passing_drop_pct           0
receiving_drop           684
receiving_drop_pct       684
passing_bad_throws         0
passing_bad_throw_pct      0
times_sacked               0
times_blitzed              0
times_hurried              0
times_hit                  0
times_pressured            0
times_pressured_pct        0
def_times_blitzed        684
def_times_hurried        684
def_times_hitqb          684
dtype: int64

In [27]:
len(pfr)

684

**Есть колонки в которых полностью отсутствуют значения, атакже эти метрики могут быть извлечены из play_by_play или tm_stats**

In [28]:
nxtgen = nfl.load_nextgen_stats(2025).to_pandas()
nxtgen.columns

Index(['season', 'season_type', 'week', 'player_display_name',
       'player_position', 'team_abbr', 'avg_time_to_throw',
       'avg_completed_air_yards', 'avg_intended_air_yards',
       'avg_air_yards_differential', 'aggressiveness',
       'max_completed_air_distance', 'avg_air_yards_to_sticks', 'attempts',
       'pass_yards', 'pass_touchdowns', 'interceptions', 'passer_rating',
       'completions', 'completion_percentage',
       'expected_completion_percentage',
       'completion_percentage_above_expectation', 'avg_air_distance',
       'max_air_distance', 'player_gsis_id', 'player_first_name',
       'player_last_name', 'player_jersey_number', 'player_short_name'],
      dtype='str')

In [29]:
nxtgen.head(2)

,season,season_type,week,player_display_name,player_position,team_abbr,avg_time_to_throw,avg_completed_air_yards,avg_intended_air_yards,avg_air_yards_differential,aggressiveness,max_completed_air_distance,avg_air_yards_to_sticks,attempts,pass_yards,pass_touchdowns,interceptions,passer_rating,completions,completion_percentage,expected_completion_percentage,completion_percentage_above_expectation,avg_air_distance,max_air_distance,player_gsis_id,player_first_name,player_last_name,player_jersey_number,player_short_name
0,2025,REG,0,Caleb Williams,QB,CHI,3.195860,6.197576,8.430951,-2.233375,10.563380,62.114966,-0.504411,568,3942,27,7,90.126174,330,58.098592,64.973275,-6.874683,21.323002,66.081562,00-0039918,Caleb,Williams,18,C.Williams
1,2025,REG,0,Matthew Stafford,QB,LAR,2.799253,7.253660,9.099604,-1.845944,18.592965,58.415054,1.016988,597,4707,46,8,109.195282,388,64.991625,63.515477,1.476147,22.215867,62.297263,00-0026498,John,Stafford,9,M.Stafford


**Идеальный датафрейм для изучения ключевых метрик игры квотербека. Не факт, что будет использовано в основной модели, но потенциально для повышения точности модели может быть применено для создания новых статистических признаков (feature engineering) для квотербека**

In [30]:
ff = nfl.load_ffverse.load_ff_opportunity(2025).to_pandas()
ff.head(2)

,season,posteam,week,game_id,player_id,full_name,position,pass_attempt,rec_attempt,rush_attempt,pass_air_yards,rec_air_yards,pass_completions,receptions,pass_completions_exp,receptions_exp,pass_yards_gained,rec_yards_gained,rush_yards_gained,pass_yards_gained_exp,rec_yards_gained_exp,rush_yards_gained_exp,pass_touchdown,rec_touchdown,rush_touchdown,pass_touchdown_exp,rec_touchdown_exp,rush_touchdown_exp,pass_two_point_conv,rec_two_point_conv,rush_two_point_conv,pass_two_point_conv_exp,rec_two_point_conv_exp,rush_two_point_conv_exp,pass_first_down,rec_first_down,rush_first_down,pass_first_down_exp,rec_first_down_exp,rush_first_down_exp,pass_interception,rec_interception,pass_interception_exp,rec_interception_exp,rec_fumble_lost,rush_fumble_lost,pass_fantasy_points_exp,rec_fantasy_points_exp,rush_fantasy_points_exp,pass_fantasy_points,rec_fantasy_points,rush_fantasy_points,total_yards_gained,total_yards_gained_exp,total_touchdown,total_touchdown_exp,total_first_down,total_first_down_exp,total_fantasy_points,total_fantasy_points_exp,pass_completions_diff,receptions_diff,pass_yards_gained_diff,rec_yards_gained_diff,rush_yards_gained_diff,pass_touchdown_diff,rec_touchdown_diff,rush_touchdown_diff,pass_two_point_conv_diff,rec_two_point_conv_diff,rush_two_point_conv_diff,pass_first_down_diff,rec_first_down_diff,rush_first_down_diff,pass_interception_diff,rec_interception_diff,pass_fantasy_points_diff,rec_fantasy_points_diff,rush_fantasy_points_diff,total_yards_gained_diff,total_touchdown_diff,total_first_down_diff,total_fantasy_points_diff,pass_attempt_team,rec_attempt_team,rush_attempt_team,pass_air_yards_team,rec_air_yards_team,pass_completions_team,receptions_team,pass_completions_exp_team,receptions_exp_team,pass_yards_gained_team,rec_yards_gained_team,rush_yards_gained_team,pass_yards_gained_exp_team,rec_yards_gained_exp_team,rush_yards_gained_exp_team,pass_touchdown_team,rec_touchdown_team,rush_touchdown_team,pass_touchdown_exp_team,rec_touchdown_exp_team,rush_touchdown_exp_team,pass_two_point_conv_team,rec_two_point_conv_team,rush_two_point_conv_team,pass_two_point_conv_exp_team,rec_two_point_conv_exp_team,rush_two_point_conv_exp_team,pass_first_down_team,rec_first_down_team,rush_first_down_team,pass_first_down_exp_team,rec_first_down_exp_team,rush_first_down_exp_team,pass_interception_team,rec_interception_team,pass_interception_exp_team,rec_interception_exp_team,rec_fumble_lost_team,rush_fumble_lost_team,pass_fantasy_points_exp_team,rec_fantasy_points_exp_team,rush_fantasy_points_exp_team,pass_fantasy_points_team,rec_fantasy_points_team,rush_fantasy_points_team,pass_completions_diff_team,receptions_diff_team,pass_yards_gained_diff_team,rec_yards_gained_diff_team,rush_yards_gained_diff_team,pass_touchdown_diff_team,rec_touchdown_diff_team,rush_touchdown_diff_team,pass_two_point_conv_diff_team,rec_two_point_conv_diff_team,rush_two_point_conv_diff_team,pass_first_down_diff_team,rec_first_down_diff_team,rush_first_down_diff_team,pass_interception_diff_team,rec_interception_diff_team,pass_fantasy_points_diff_team,rec_fantasy_points_diff_team,rush_fantasy_points_diff_team,total_yards_gained_team,total_yards_gained_exp_team,total_yards_gained_diff_team,total_touchdown_team,total_touchdown_exp_team,total_touchdown_diff_team,total_first_down_team,total_first_down_exp_team,total_first_down_diff_team,total_fantasy_points_team,total_fantasy_points_exp_team,total_fantasy_points_diff_team
0,2025,ARI,1.0,2025_01_ARI_NO,00-0035228,Kyler Murray,QB,29.0,0.0,7.0,166.0,0.0,21.0,0.0,21.63,0.00,163.0,0.0,38.0,217.9,0.00,41.4,2.0,0.0,0.0,1.9,0.00,0.4,0.0,0.0,0.0,0.0,0.0,0.0,11.0,0.0,3.0,10.54,0.00,2.63,0.0,0.0,0.42,0.00,0.0,0.0,15.47,0.00,6.54,14.52,0.0,3.8,201.0,259.30,2.0,2.30,14.0,13.17,18.32,22.01,-0.63,0.00,-54.9,0.00,-3.4,0.1,0.00,-0.4,0.0,0.0,0.0,0.46,0.00,0.37,-0.42,0.00,-0.95,0.00,-2.74,-58.30,-0.30,0.83,-3.69,29.0,29.0,27.0,166.0,166.0,21.0,21.0,21.63,21.62,163.0,163.0,146.0,217.9,217.91,131.73,2.0,2.0,0.0,1.9,1.89,1.28,0.0,0.0,0.0,0.0,0

In [31]:
ff.info()

<class 'pandas.DataFrame'>
RangeIndex: 6054 entries, 0 to 6053
Columns: 159 entries, season to total_fantasy_points_diff_team
dtypes: float64(153), str(6)
memory usage: 7.6 MB


**Фэнтэзи очки в чемпионатах для фанатов излишни для модели предсказания результатов матчей. Поэтому все данные из модуля load_ffverse, которые предоставляют статистику из фэнтэзи-лиг, не будут использованы**

**Данные о контрактах, результатов испытаний на драфт-комбайне, трейдах игроков, а также драфт-пиках находящихся в распоряжении команд, не несут в себе пользы для предсказания итогов матчей**

In [32]:
combine = nfl.load_combine([2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025])
combine = combine.to_pandas()
combine.info()

<class 'pandas.DataFrame'>
RangeIndex: 3425 entries, 0 to 3424
Data columns (total 18 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   season       3425 non-null   int32  
 1   draft_year   2128 non-null   float64
 2   draft_team   2128 non-null   str    
 3   draft_round  2128 non-null   float64
 4   draft_ovr    2128 non-null   float64
 5   pfr_id       3026 non-null   str    
 6   cfb_id       3180 non-null   str    
 7   player_name  3425 non-null   str    
 8   pos          3425 non-null   str    
 9   school       3425 non-null   str    
 10  ht           3396 non-null   str    
 11  wt           3401 non-null   float64
 12  forty        2674 non-null   float64
 13  bench        1859 non-null   float64
 14  vertical     2607 non-null   float64
 15  broad_jump   2539 non-null   float64
 16  cone         1624 non-null   float64
 17  shuttle      1726 non-null   float64
dtypes: float64(10), int32(1), str(7)
memory usage: 694.4 KB


In [33]:
combine.head()

,season,draft_year,draft_team,draft_round,draft_ovr,pfr_id,cfb_id,player_name,pos,school,ht,wt,forty,bench,vertical,broad_jump,cone,shuttle
0,2016,NaN,NaN,NaN,NaN,AbdeMe00,mehdi-abdesmad-1,Mehdi Abdesmad,DE,Boston Col.,6-6,284.0,5.10,25.0,29.5,108.0,7.55,4.62
1,2016,NaN,NaN,NaN,NaN,NaN,vernon-adams-1,Vernon Adams,QB,Oregon,5-11,200.0,4.83,NaN,29.5,114.0,6.82,4.20
2,2016,2016.0,New York Giants,6.0,184.0,AdamJe01,jerell-adams-1,Jerell Adams,TE,South Carolina,6-5,247.0,4.64,NaN,32.5,117.0,7.05,4.31
3,2016,NaN,NaN,NaN,NaN,AddiBr01,bralon-addison-1,Bralon Addison,WR,Oregon,5-9,197.0,4.66,13.0,34.5,116.0,6.95,4.14
4,2016,2016.0,Tampa Bay Buccaneers,2.0,59.0,AguaRo00,roberto-aguayo-1,Roberto Aguayo,K,Florida State,6-0,207.0,4.96,NaN,NaN,NaN,NaN,NaN


In [34]:
contracts = nfl.load_contracts().to_pandas()

In [35]:
contracts.head(1)

,player,position,team,is_active,year_signed,years,value,apy,guaranteed,apy_cap_pct,inflated_value,inflated_apy,inflated_guaranteed,player_page,otc_id,gsis_id,height,weight,college,draft_year,draft_round,draft_overall,draft_team,date_of_birth,season_history,contract_history
0,Joe Burrow,QB,Bengals,True,2023,5.0,275.0,55.0,146.51,0.245,368.460854,73.692171,196.302544,https://overthecap.com/player/joe-burrow/8741/,8741,00-0036442,"6'4""",215,LSU,2020.0,1.0,1.0,Bengals,NaN,"[{'year': '2020', 'team': 'Bengals', 'base_sal...","[{'team': 'Bengals', 'contract_type': 'Drafted..."


In [36]:
dp = nfl.load_draft_picks([2026]).to_pandas()

In [37]:
dp.info()

<class 'pandas.DataFrame'>
RangeIndex: 257 entries, 0 to 256
Data columns (total 36 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   season            257 non-null    int32  
 1   round             257 non-null    int32  
 2   pick              257 non-null    int32  
 3   team              257 non-null    str    
 4   gsis_id           230 non-null    str    
 5   pfr_player_id     257 non-null    str    
 6   cfb_player_id     205 non-null    str    
 7   pfr_player_name   257 non-null    str    
 8   hof               257 non-null    bool   
 9   position          257 non-null    str    
 10  category          257 non-null    str    
 11  side              256 non-null    str    
 12  college           257 non-null    str    
 13  age               215 non-null    float64
 14  to                0 non-null      float64
 15  allpro            257 non-null    int32  
 16  probowls          257 non-null    int32  
 17  seasons_

In [38]:
dp.head(1)

,season,round,pick,team,gsis_id,pfr_player_id,cfb_player_id,pfr_player_name,hof,position,category,side,college,age,to,allpro,probowls,seasons_started,w_av,car_av,dr_av,games,pass_completions,pass_attempts,pass_yards,pass_tds,pass_ints,rush_atts,rush_yards,rush_tds,receptions,rec_yards,rec_tds,def_solo_tackles,def_ints,def_sacks
0,2026,1,1,LVR,MEN516487,MendFe00,fernando-mendoza-1,Fernando Mendoza,False,QB,QB,O,Indiana,22.0,NaN,0,0,0,NaN,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [39]:
trades = nfl.load_trades().to_pandas()
trades.head(2)

,trade_id,season,trade_date,gave,received,pick_season,pick_round,pick_number,conditional,pfr_id,pfr_name
0,701,2002,2002-03-04,HOU,WAS,NaN,NaN,NaN,NaN,WuerDa00,Danny Wuerffel
1,701,2002,2002-03-04,WAS,HOU,NaN,NaN,NaN,NaN,DeLoJe20,Jerry DeLoach


**Ниже предсталвен датафрейм teams, которые является справочником, который содержит данные о командах**

In [40]:
teams = nfl.load_teams().to_pandas()
teams.head(2)

,team_abbr,team_name,team_id,team_nick,team_conf,team_division,team_color,team_color2,team_color3,team_color4,team_logo_wikipedia,team_logo_espn,team_wordmark,team_conference_logo,team_league_logo,team_logo_squared
0,ARI,Arizona Cardinals,3800,Cardinals,NFC,NFC West,#97233F,#000000,#ffb612,#a5acaf,https://upload.wikimedia.org/wikipedia/en/thum...,https://a.espncdn.com/i/teamlogos/nfl/500/ari.png,https://github.com/nflverse/nflverse-pbp/raw/m...,https://github.com/nflverse/nflverse-pbp/raw/m...,https://raw.githubusercontent.com/nflverse/nfl...,https://github.com/nflverse/nflverse-pbp/raw/m...
1,ATL,Atlanta Falcons,0200,Falcons,NFC,NFC South,#A71930,#000000,#a5acaf,#a30d2d,https://upload.wikimedia.org/wikipedia/en/thum...,https://a.espncdn.com/i/teamlogos/nfl/500/atl.png,https://github.com/nflverse/nflverse-pbp/raw/m...,https://github.com/nflverse/nflverse-pbp/raw/m...,https://raw.githubusercontent.com/nflverse/nfl...,https://github.com/nflverse/nflverse-pbp/raw/m...


In [41]:
teams.info()

<class 'pandas.DataFrame'>
RangeIndex: 36 entries, 0 to 35
Data columns (total 16 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   team_abbr             36 non-null     str  
 1   team_name             36 non-null     str  
 2   team_id               36 non-null     str  
 3   team_nick             36 non-null     str  
 4   team_conf             36 non-null     str  
 5   team_division         36 non-null     str  
 6   team_color            36 non-null     str  
 7   team_color2           36 non-null     str  
 8   team_color3           34 non-null     str  
 9   team_color4           34 non-null     str  
 10  team_logo_wikipedia   36 non-null     str  
 11  team_logo_espn        36 non-null     str  
 12  team_wordmark         36 non-null     str  
 13  team_conference_logo  36 non-null     str  
 14  team_league_logo      36 non-null     str  
 15  team_logo_squared     36 non-null     str  
dtypes: str(16)
memory usa

**По итогам анализа библиотеки nflreadpy, следующие датафреймы будут использованы для создания новых признаков (feature engineering) и на основе которых будет строится модель:**
1. schedules
2. play_by_play
3. rosters_weekly
4. injuries
5. tm_stats
6. nxtgen
7. teams 
?. players, depth_charts

**При этом загружать их и сохранять как csv нет необходимости, так как эти данные обновляются в библиотеке ежедневно, что пригодится для каждой новой игровой недели. Только NFL_schedule был сохранен, так как расписание сезона уже известно**
